In [1]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 重新拉取沪深300成分股收盘价
hs300 = ak.index_stock_cons_weight_csindex(symbol="000300")
stock_list = hs300["成分券代码"].tolist()

close_dict = {}
for i, code in enumerate(stock_list):
    try:
        df = ak.stock_zh_a_hist(symbol=code, adjust="qfq",
                                start_date="2023-01-01")
        close_dict[code] = df.set_index("日期")["收盘"]
        if i % 50 == 0:
            print(f"进度：{i}/300")
    except:
        continue

close_df = pd.DataFrame(close_dict)
close_df.index = pd.to_datetime(close_df.index)
close_df = close_df[close_df.index >= "2023-01-01"]
close_df = close_df.dropna(axis=1, thresh=len(close_df)*0.9)

print(f"\n数据加载完成：{close_df.shape}")

进度：0/300
进度：50/300
进度：100/300
进度：150/300
进度：200/300
进度：250/300

数据加载完成：(781, 296)


In [2]:
# 存成CSV，以后直接读本地
close_df.to_csv("close_data.csv")
print("已保存到 close_data.csv")

已保存到 close_data.csv


In [3]:
close_df = pd.read_csv("close_data.csv", index_col=0, parse_dates=True)

In [4]:
# 计算三个因子
momentum = close_df / close_df.shift(20) - 1
reversal = -(close_df / close_df.shift(5) - 1)

# 拉取换手率数据
volume_dict = {}
for i, code in enumerate(close_df.columns):
    try:
        df = ak.stock_zh_a_hist(symbol=code, adjust="qfq",
                                start_date="2023-01-01")
        volume_dict[code] = df.set_index("日期")["换手率"]
        if i % 50 == 0:
            print(f"进度：{i}/{len(close_df.columns)}")
    except:
        continue

volume_df = pd.DataFrame(volume_dict)
volume_df.index = pd.to_datetime(volume_df.index)
volume_df = volume_df[volume_df.index >= "2023-01-01"]
volume_df.to_csv("volume_data.csv")
print("换手率数据已保存")

进度：0/296
进度：50/296
进度：100/296
进度：150/296
进度：200/296
进度：250/296
换手率数据已保存


In [5]:
# 标准化函数
def standardize(factor):
    return (factor - factor.mean(axis=1).values.reshape(-1,1)) / \
            factor.std(axis=1).values.reshape(-1,1)

# 三因子标准化
mom_std = standardize(momentum)
rev_std = standardize(reversal)

volume_df2 = pd.read_csv("volume_data.csv", index_col=0, parse_dates=True)
turnover_factor = -volume_df2.rolling(20).mean()
turn_std = standardize(turnover_factor)

# 等权合成
combined = (mom_std + rev_std + turn_std) / 3

# 每月第一个交易日调仓，选因子排名前20%的股票
def get_selected_stocks(factor_row, top_pct=0.2):
    valid = factor_row.dropna()
    n = max(int(len(valid) * top_pct), 1)
    return valid.nlargest(n).index.tolist()

print("合成因子计算完成")
print(f"因子维度：{combined.shape}")

合成因子计算完成
因子维度：(781, 296)


In [7]:
# 回测参数
rebalance_freq = 20      # 每20个交易日调仓一次
top_pct = 0.2            # 选前20%的股票
cost = 0.002             # 每次交易成本0.2%

# 初始化
portfolio_value = [1.0]  # 初始净值为1
current_stocks = []      # 当前持仓

# 找到所有调仓日（每20个交易日）
dates = combined.index.tolist()
rebalance_dates = dates[20::rebalance_freq]

for i, date in enumerate(dates[20:], start=20):
    # 调仓日：重新选股
    if date in rebalance_dates:
        factor_row = combined.loc[date]
        current_stocks = get_selected_stocks(factor_row, top_pct)

    # 计算当天组合收益
    if len(current_stocks) == 0:
        portfolio_value.append(portfolio_value[-1])
        continue

    # 持仓股票当天的等权收益
    prev_date = dates[i-1]
    returns = []
    for code in current_stocks:
        if code in close_df.columns:
            try:
                r = close_df.loc[date, code] / close_df.loc[prev_date, code] - 1
                if not np.isnan(r):
                    returns.append(r)
            except:
                continue

    if len(returns) == 0:
        portfolio_value.append(portfolio_value[-1])
        continue

    daily_return = np.mean(returns)

    # 调仓日扣除交易成本
    if date in rebalance_dates:
        daily_return -= cost

    portfolio_value.append(portfolio_value[-1] * (1 + daily_return))

portfolio = pd.Series(portfolio_value, index=dates[20-1:])
print(f"回测完成，最终净值：{portfolio.iloc[-1]:.4f}")

回测完成，最终净值：1.0856
